In [8]:
import cv2
import torch
import numpy as np

from datasets_loader.asl_dataset import ASLDataset
from model import SignModel
from hand_detection import HandDetection

checkpoint = torch.load("model.pth", map_location="cpu")
num_classes = checkpoint["num_classes"]

model = SignModel(num_classes=num_classes)
model.load_state_dict(checkpoint["model_state"])
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

hand_detection = HandDetection()
dataset = ASLDataset("datasets/ASL_Alphabet_Dataset/train")
label_map = dataset.label_map

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    detection_result = hand_detection.detect_video(rgb_frame)

    predicted_label = None

    if detection_result.hand_landmarks:
        hand_landmarks = detection_result.hand_landmarks[0]

        landmarks_array = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks])

        wrist = landmarks_array[0]

        landmarks_rel = landmarks_array - wrist

        landmarks_tensor = torch.tensor(landmarks_rel.flatten(), dtype=torch.float32).unsqueeze(0).to(device)

        # Predict sign
        with torch.no_grad():
            output = model(landmarks_tensor)
            pred_class = torch.argmax(output, dim=1).item()
            predicted_label = label_map[pred_class]

    annotated_frame = hand_detection.draw_landmarks_on_image(
        rgb_frame, detection_result, predicted_label
    )

    annotated_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)

    cv2.imshow("ASL Sign Detection", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('x'):
        break

cap.release()
cv2.destroyAllWindows()
hand_detection.close()

I0000 00:00:1774323260.086970   43465 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1774323260.095578   43477 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: AMD Radeon Vega 10 Graphics (radeonsi, raven, ACO, DRM 3.64, 6.17.0-19-generic)
W0000 00:00:1774323260.116786   43470 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774323260.144085   43466 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1774323260.150959   43478 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1774323260.155368   43490 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: AMD Radeon Vega 10 Graphics (radeonsi, raven, ACO, DRM 3.64, 6.17.0-19-generic)
